# QGNN Triple-Shield — reproducible entry point

Tests whether a **permutation-equivariant** quantum jet tagger is a *triple shield*: generalizes from **fewer samples**, works at **fewer qubits**, and tolerates **more (correlated) noise** — vs a budget-matched non-equivariant control and classical PFN baselines, on JetNet **gluon vs top**.

Run cells top-to-bottom. The cheap parts (data check, unit tests, plots from cached results) run in minutes; the **full sweep** (~3.5–4 h on ~8–10 CPU cores) is launched from a terminal, not inline.

## 1. Setup (platform-aware)

In [ ]:
import os, sys, subprocess

def detect():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ: return 'kaggle'
    try:
        import google.colab  # noqa
        return 'colab'
    except ImportError:
        return 'local'

PLATFORM = detect()
print('platform:', PLATFORM)
os.environ['PYTHONUTF8'] = '1'  # Windows/jetnet progress-bar safety

if PLATFORM in ('colab', 'kaggle'):
    # NumPy pinned <2 (jetnet->energyflow->wasserstein); then the rest.
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'numpy<2', 'pennylane', 'pennylane-lightning', 'jetnet',
                    'pyyaml', 'pyarrow', 'scikit-learn', 'matplotlib'], check=True)
    if PLATFORM == 'colab':
        from google.colab import drive; drive.mount('/content/drive')
    print('Restart the runtime now if numpy was downgraded, then re-run from here.')

# Make sure we are at the repo root (folder containing config.yaml).
import glob
if not os.path.exists('config.yaml'):
    hits = glob.glob('**/config.yaml', recursive=True)
    if hits: os.chdir(os.path.dirname(os.path.abspath(hits[0])))
print('cwd:', os.getcwd())
sys.path.insert(0, os.getcwd())

## 2. Data check — shapes, class balance, example jets

In [ ]:
!python scripts/check_data.py

## 3. Scientific-correctness unit tests
Permutation invariance (equivariant) vs non-invariance (control), budget match, and training plumbing.

In [ ]:
!python tests/test_loader_logic.py
!python tests/test_invariance.py
!python tests/test_overfit.py

## 4. Smoke sweep (a few minutes)
End-to-end on a tiny grid, with per-config caching. Confirms the full pipeline before the long run.

In [ ]:
!python experiments/sweep.py --grid smoke --workers 4

## 5. Full sweep + diagnostics  (run in a TERMINAL, ~3.5–4 h)
Not run inline (long). In a terminal at the repo root:
```bash
python experiments/sweep.py --grid full --workers 9   # resumes if interrupted (cached)
python experiments/run_diagnostics.py                 # barren-plateau curves (~1 min)
```
Results land in `results/results_full.parquet` and `results/diagnostics_gradvar.parquet`.

## 6. Figures (from cached results)

In [ ]:
# Plot from full results if present, else fall back to the smoke results.
import os
res = 'results/results_full.parquet' if os.path.exists('results/results_full.parquet') else 'results/results_smoke.parquet'
print('plotting from', res)
subprocess_res = os.system(f'python analysis/plots.py --results {res}')

In [ ]:
from IPython.display import Image, display
import glob
for png in sorted(glob.glob('results/figures/*.png')):
    print(png)
    display(Image(png))